# LM Arena Rankings: Predicting AI Model ELO Ratings

**What this notebook covers:**
- Loading and exploring the LM Arena (Chatbot Arena) leaderboard dataset
- Cleaning data and checking for data leakage
- Feature engineering (dates, high-cardinality encoding)
- Training and comparing 3 ML models (Random Forest, XGBoost, LightGBM) to predict a model's Arena `rating` (Elo score)
- Evaluating models honestly with R², RMSE, MAE
- Final conclusion with key takeaways

**Goal:** Predict the Arena `rating` of an AI model using only information that would realistically be known beforehand (organization, license, category/subset, vote count, date) — *without* leaking the target.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print("Libraries loaded successfully!")


## 2. Load the Dataset

We load the CSV and take a first look at its shape, columns, and data types.

In [ ]:
df = pd.read_csv('/kaggle/input/ai-model-arena-rankings/ai_model_arena_rankings_csv.csv')

print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


## 3. Exploratory Data Analysis (EDA)

Let's check missing values, duplicates, and understand what each column means.

**Column notes (from inspection):**
- `rating` → the Elo-style Arena score (our **target**)
- `rating_lower`, `rating_upper`, `variance` → confidence interval built directly FROM `rating` → **leakage, must drop**
- `rank` → is basically `rating` sorted → **leakage, must drop**
- `model_name` → free text, one row per model per date, too high-cardinality to use directly
- `organization`, `license`, `subset` → useful categorical features
- `vote_count` → useful numeric feature
- `leaderboard_publish_date` → useful for extracting time-based features
- `category` → always "overall", no information → drop


In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nUnique models:", df['model_name'].nunique())
print("\nUnique organizations:", df['organization'].nunique())
print("\nSubset counts:\n", df['subset'].value_counts())


In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['rating'], bins=50, kde=True, color='teal')
plt.title('Distribution of Arena Ratings')
plt.xlabel('Rating (Elo)')
plt.show()


In [ ]:
top_orgs = df['organization'].value_counts().head(10)
plt.figure(figsize=(8,4))
sns.barplot(x=top_orgs.values, y=top_orgs.index, palette='viridis')
plt.title('Top 10 Organizations by Number of Leaderboard Entries')
plt.xlabel('Count')
plt.show()


## 4. Data Leakage Check

`rank`, `rating_lower`, `rating_upper`, and `variance` are all mathematically derived from `rating` itself
(rank = ordering of rating; lower/upper = rating ± confidence bound based on variance).
Using them as features would let the model "cheat", so we drop them along with `category` (constant column).


In [ ]:
leakage_cols = ['rank', 'rating_lower', 'rating_upper', 'variance', 'category']
df_clean = df.drop(columns=leakage_cols)

# Drop rows with missing vote_count / organization / license (small % of data)
df_clean = df_clean.dropna(subset=['vote_count', 'organization', 'license'])
print("Shape after cleaning:", df_clean.shape)


## 5. Feature Engineering

- Convert `leaderboard_publish_date` into `year` and `month` (captures how models improve over time)
- `organization` and `license` are high-cardinality categorical columns → use **frequency (count) encoding**,
  a simple, beginner-friendly, and leakage-safe way to represent them numerically
- `subset` has only 5 categories → one-hot encode it


In [ ]:
df_clean['leaderboard_publish_date'] = pd.to_datetime(df_clean['leaderboard_publish_date'])
df_clean['year'] = df_clean['leaderboard_publish_date'].dt.year
df_clean['month'] = df_clean['leaderboard_publish_date'].dt.month

# Frequency encoding for high-cardinality columns
org_freq = df_clean['organization'].value_counts()
df_clean['organization_freq'] = df_clean['organization'].map(org_freq)

license_freq = df_clean['license'].value_counts()
df_clean['license_freq'] = df_clean['license'].map(license_freq)

# One-hot encode subset (low cardinality)
df_clean = pd.get_dummies(df_clean, columns=['subset'], prefix='subset')

df_clean.head()


In [ ]:
feature_cols = ['vote_count', 'year', 'month', 'organization_freq', 'license_freq'] + \
               [c for c in df_clean.columns if c.startswith('subset_')]

X = df_clean[feature_cols]
y = df_clean['rating']

print("Features used:", feature_cols)
print("X shape:", X.shape, "| y shape:", y.shape)


## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


## 7. Train & Compare Models

We train 3 popular regression models and compare them honestly on the same test set:
- Random Forest
- XGBoost
- LightGBM


In [ ]:
models = {
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    results.append({"Model": name, "R2 Score": r2, "RMSE": rmse, "MAE": mae})
    print(f"{name:15s} -> R2: {r2:.4f} | RMSE: {rmse:.2f} | MAE: {mae:.2f}")


## 8. Compare Results

In [ ]:
results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False).reset_index(drop=True)
results_df


In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(data=results_df, x='Model', y='R2 Score', palette='crest')
plt.title('Model Comparison - R2 Score (higher is better)')
plt.ylim(0, 1)
plt.show()


## 9. Feature Importance (Best Model)

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette='mako')
plt.title(f'Feature Importance - {best_model_name}')
plt.xlabel('Importance')
plt.show()


## 10. Conclusion

- We predicted an AI model's **Arena Elo rating** using only pre-known, non-leaking features:
  organization, license, subset (modality), vote count, and publish date.
- All potential **leakage columns** (`rank`, `rating_lower`, `rating_upper`, `variance`) were correctly
  identified and dropped before modeling.
- Among the three models tested, **tree-based ensemble methods performed best**, with the top model
  achieving the highest R² score shown in the comparison table above — a realistic, honest result
  given the modest feature set (no leakage-based shortcuts were used).
- `vote_count` and `organization_freq` were consistently among the most important features, suggesting
  that **popularity/exposure (votes)** and **which lab built the model** are strong signals of its
  perceived quality on the Arena leaderboard.
- **Next steps:** this can be improved further by adding NLP-based features from `model_name`
  (e.g. model family, parameter size, release recency), or by treating this as a **time-series
  problem** per model to track rating trends over successive leaderboard snapshots.
